# Invoice reconciliation on a Fabric Lakehouse

Runs the repository's Bronze → Silver → Gold pipeline from a notebook, using the same
`invoice_recon` package as the local CLI. All data is synthetic.

**Status: execution inside Microsoft Fabric has NOT been tested.** The code cells below are
executed locally by `tests/test_notebook.py` against a local Spark + Delta session; the
Fabric-specific parts (the `%pip` cell, `/lakehouse/default`, relative `Files/` and `Tables/`
paths) are written from the Fabric documentation and have not been run in a Fabric capacity.

## Setup in Fabric (fictional names)

1. In a workspace backed by a Fabric capacity or trial (for example `ws-portfolio-sandbox`),
   create a Lakehouse named `lh_invoice_recon` **without** lakehouse schemas, so that tables
   live directly under `Tables/`.
2. Build the wheel from the repository root: `python -m pip wheel . --no-deps -w dist`.
3. Upload `dist/fabric_invoice_reconciliation-0.1.0-py3-none-any.whl` to the lakehouse folder
   `Files/libs/`.
4. Import this notebook, attach `lh_invoice_recon` as its default lakehouse and use the
   Runtime 2.0 environment (Spark 4.1, Delta Lake 4.2, Python 3.13).
5. Run the cells in order. Do not install `pyspark` or `delta-spark`: the runtime provides
   them, which is why the wheel is installed with `--no-deps`.


In [ ]:
%pip install /lakehouse/default/Files/libs/fabric_invoice_reconciliation-0.1.0-py3-none-any.whl --no-deps

In [ ]:
# Parameters. In Fabric keep these defaults; the local test overrides them.
LAKEHOUSE_FILES_DIR = "/lakehouse/default"  # file-API mount of the default lakehouse (the generator writes here)
LAKEHOUSE_SPARK_ROOT = ""                   # "" -> Spark paths resolve as Files/... and Tables/... in the default lakehouse
SEED = 4242
BATCHES = 2
INVOICES_PER_BATCH = 5000
START_PERIOD = "2026-03"


In [ ]:
from pathlib import Path

from invoice_recon import generator, pipeline
from invoice_recon.config import Lakehouse, load_rules
from invoice_recon.report import build_report, render_markdown
from invoice_recon.spark import configure_session
from invoice_recon.validate import validate_batch

configure_session(spark)  # `spark` is the notebook's session: ANSI mode on, UTC session time zone
lakehouse = Lakehouse(LAKEHOUSE_SPARK_ROOT)
rules = load_rules()
print("Spark", spark.version, "| rules", rules.version)


## 1. Synthetic landing data

Writes `Files/landing/<batch>/invoices.csv`, `ledger.jsonl` and `batch.json`, plus the designed
outcome of every record under `Files/synthetic_ground_truth/`. Existing batches are left as they are.

In [ ]:
landing = Path(LAKEHOUSE_FILES_DIR) / "Files" / "landing"
existing = sorted(p.parent.name for p in landing.glob("*/batch.json"))
if existing:
    print("landing data already present:", existing)
else:
    for summary in generator.generate(LAKEHOUSE_FILES_DIR, seed=SEED, batches=BATCHES,
                                      invoices=INVOICES_PER_BATCH, start_period=START_PERIOD):
        print(summary.batch_id, summary.invoice_rows, "invoice rows,", summary.ledger_rows, "ledger rows")


## 2. Bronze → Silver → Gold

Each stage replaces its batch slice in every table it owns (`replaceWhere` on `_batch_id`), so
running the cell twice gives the same tables.

In [ ]:
results = pipeline.run(spark, lakehouse, rules=rules)
batch_ids = sorted({result.batch_id for result in results})


## 3. Record conservation and invariants

In [ ]:
for batch_id in batch_ids:
    validation = validate_batch(spark, lakehouse, batch_id)
    print(batch_id, "PASSED" if validation.passed else "FAILED")
    for check in validation.checks:
        print(f"  {'ok  ' if check.passed else 'FAIL'} {check.check_id}: {check.detail}")


## 4. Query the Delta tables

Tables are path-based Delta tables under `Tables/`. In a Fabric lakehouse they should also appear
in the SQL analytics endpoint (not tested).

In [ ]:
gold = spark.read.format("delta").load(lakehouse.table("gold_reconciliation"))
gold.groupBy("_batch_id", "side", "status").count().orderBy("_batch_id", "side", "status").show(50, truncate=False)


In [ ]:
evidence_path = lakehouse.table("gold_match_evidence")
spark.sql(f"""
    SELECT decision, reason_code, count(*) AS records
    FROM delta.`{evidence_path}`
    GROUP BY decision, reason_code
    ORDER BY records DESC
""").show(truncate=False)


In [ ]:
exceptions = spark.read.format("delta").load(lakehouse.table("gold_exceptions"))
exceptions.groupBy("priority", "status").count().orderBy("priority", "status").show(truncate=False)
exceptions.orderBy("priority", "side", "line_number").select(
    "side", "line_number", "status", "reason_code", "recommended_action").show(10, truncate=False)


## 5. Reading the results

* **Status** (one per source record): `MATCHED`, `AMOUNT_MISMATCH`, `CURRENCY_MISMATCH`,
  `ONLY_IN_INVOICES`, `ONLY_IN_LEDGER`, `AMBIGUOUS`, `DUPLICATE`, `INVALID_RECORD`.
* **Flags** add detail. `AMBIGUOUS_DATE`, `PERIOD_INCONSISTENT` and `OUT_OF_PERIOD` send even a
  matched record to review; `REFERENCE_FORMAT_VARIANT` and `ROUNDING_DIFFERENCE` are informational.
* **gold_match_evidence** explains every decided pair and every undecided candidate set.
* **gold_candidate_suggestions** are similarity-based hints for unmatched records. They are
  never applied automatically.
* **gold_exceptions** is the review queue with a priority and a recommended action.

In [ ]:
report = build_report(spark, lakehouse, batch_ids[0], validate_batch(spark, lakehouse, batch_ids[0]))
print(render_markdown(report)[:4000])


In [ ]:
spark.sql(f"DESCRIBE HISTORY delta.`{lakehouse.table('gold_reconciliation')}`") \
    .select("version", "operation", "operationParameters").show(truncate=False)
